In [1]:
# 05_alarms_safety.ipynb – Ventilator Alarm and Safety System

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ============================================================
# Lung model (single compartment)
# ============================================================

def lung_ode(t, y, R, C, P_insp, PEEP, insp_time, rate):
    V = y[0]
    period = 60 / rate
    t_mod = t % period
    P_aw = P_insp if t_mod < insp_time else 0.0
    Q = (P_aw - V/C) / R
    if V <= 0 and Q < 0:
        Q = 0
    return [Q]

def simulate_lung(R, C, P_insp, PEEP, rate, insp_frac=0.33, duration=10):
    insp_time = (60/rate) * insp_frac
    t_eval = np.linspace(0, duration, 2000)
    sol = solve_ivp(lung_ode, (0, duration), [0.0], t_eval=t_eval,
                    args=(R, C, P_insp, PEEP, insp_time, rate),
                    method='RK45', rtol=1e-6)
    t = sol.t
    V = sol.y[0]
    Q = np.gradient(V, t)
    # Recompute P_aw
    P_aw = np.zeros_like(t)
    for i, ti in enumerate(t):
        period = 60/rate
        t_mod = ti % period
        P_aw[i] = P_insp if t_mod < insp_time else 0.0
    P_aw += PEEP + V/C + R*Q   # full equation
    return t, V, Q, P_aw

# ============================================================
# Alarm detection functions
# ============================================================

def detect_alarms(t, V, Q, P_aw, rate, alarms):
    """
    alarms is a dict with thresholds and flags.
    Returns updated alarms dict with boolean flags and messages.
    """
    # 1. High peak pressure
    peak_P = np.max(P_aw)
    alarms['high_pressure_trigger'] = peak_P > alarms['high_pressure_threshold']
    alarms['high_pressure_msg'] = f"Peak pressure {peak_P:.1f} cmH2O exceeds {alarms['high_pressure_threshold']} cmH2O" if alarms['high_pressure_trigger'] else ""
    
    # 2. Low tidal volume (over last breath)
    # Find breaths by zero‑crossings of flow (inspiratory start)
    zero_crossings = []
    for i in range(1, len(Q)):
        if Q[i-1] <= 0 and Q[i] > 0:
            zero_crossings.append(i)
    if len(zero_crossings) >= 2:
        # last full breath from last crossing to previous crossing
        idx_start = zero_crossings[-2]
        idx_end = zero_crossings[-1]
        Vt = (V[idx_end] - V[idx_start]) * 1000  # mL
        alarms['low_Vt_trigger'] = Vt < alarms['low_Vt_threshold']
        alarms['low_Vt_msg'] = f"Last tidal volume {Vt:.0f} mL below {alarms['low_Vt_threshold']} mL" if alarms['low_Vt_trigger'] else ""
    else:
        alarms['low_Vt_trigger'] = False
        alarms['low_Vt_msg'] = ""
    
    # 3. Apnea (no breath for > apnea_time seconds)
    # Check time since last zero crossing
    current_time = t[-1]
    if zero_crossings:
        last_breath_time = t[zero_crossings[-1]]
        time_since_last_breath = current_time - last_breath_time
    else:
        time_since_last_breath = current_time
    alarms['apnea_trigger'] = time_since_last_breath > alarms['apnea_time']
    alarms['apnea_msg'] = f"No breath for {time_since_last_breath:.1f} seconds" if alarms['apnea_trigger'] else ""
    
    # 4. Disconnection (very low pressure and no flow change)
    # Simple: if P_aw near 0 and Q near 0 for a sustained period
    low_pressure_threshold = 2.0
    low_flow_threshold = 0.1
    disconnection_duration = 0.5  # seconds
    low_pressure_streak = 0
    for i in range(len(t)-1, -1, -1):
        if P_aw[i] < low_pressure_threshold and abs(Q[i]) < low_flow_threshold:
            low_pressure_streak += (t[i] - t[i-1]) if i>0 else 0
        else:
            break
    alarms['disconnect_trigger'] = low_pressure_streak > disconnection_duration
    alarms['disconnect_msg'] = "Patient disconnected!" if alarms['disconnect_trigger'] else ""
    
    return alarms

def plot_with_alarms(t, V, Q, P_aw, R, C, P_insp, PEEP, rate, insp_frac, alarms):
    # Plot all waveforms
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
    ax1, ax2 = axes[0, 0], axes[0, 1]
    ax3, ax4 = axes[1, 0], axes[1, 1]
    
    ax1.plot(t, P_aw, 'b-', lw=2)
    ax1.axhline(PEEP, color='gray', linestyle='--', label=f'PEEP = {PEEP}')
    if alarms['high_pressure_threshold']:
        ax1.axhline(alarms['high_pressure_threshold'], color='red', linestyle='--', label='High P limit')
    ax1.set_ylabel('Pressure (cmH2O)')
    ax1.set_title('Airway Pressure')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(t, Q, 'g-', lw=2)
    ax2.axhline(0, color='k', lw=0.5)
    ax2.set_ylabel('Flow (L/s)')
    ax2.set_title('Flow')
    ax2.grid(True, alpha=0.3)
    
    ax3.plot(t, V*1000, 'r-', lw=2)
    if alarms['low_Vt_threshold']:
        ax3.axhline(alarms['low_Vt_threshold'], color='orange', linestyle='--', label=f'Low Vt limit {alarms["low_Vt_threshold"]} mL')
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Volume (mL)')
    ax3.set_title('Lung Volume')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Pressure-Volume loop
    ax4.plot(P_aw, V*1000, 'purple', lw=1.5)
    ax4.set_xlabel('Pressure (cmH2O)')
    ax4.set_ylabel('Volume (mL)')
    ax4.set_title('Pressure-Volume Loop')
    ax4.grid(True, alpha=0.3)
    
    # Construct alarm text
    alarm_msgs = []
    if alarms['high_pressure_trigger']:
        alarm_msgs.append(f"⚠️ {alarms['high_pressure_msg']}")
    if alarms['low_Vt_trigger']:
        alarm_msgs.append(f"⚠️ {alarms['low_Vt_msg']}")
    if alarms['apnea_trigger']:
        alarm_msgs.append(f"⚠️ {alarms['apnea_msg']}")
    if alarms['disconnect_trigger']:
        alarm_msgs.append(f"🚨 {alarms['disconnect_msg']}")
    if not alarm_msgs:
        alarm_msgs.append("✅ No active alarms")
    
    alarm_text = "\n".join(alarm_msgs)
    fig.suptitle(f'Ventilator Alarms | R={R:.1f}, C={C:.3f}, Rate={rate} bpm')
    ax1.text(0.02, 0.98, alarm_text, transform=ax1.transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightcoral' if alarms['disconnect_trigger'] else 'lightyellow', alpha=0.8))
    
    plt.show()
    
    # Also print to console
    print("\n🔔 Alarm Status:")
    for msg in alarm_msgs:
        print(f"  {msg}")

# ============================================================
# Interactive widgets
# ============================================================

style = {'description_width': 'initial'}

# Lung and ventilator settings
R_slider = widgets.FloatSlider(value=8, min=2, max=30, step=0.5, description='Resistance (cmH2O/L/s):', style=style)
C_slider = widgets.FloatSlider(value=0.06, min=0.02, max=0.15, step=0.002, description='Compliance (L/cmH2O):', style=style)
P_insp_slider = widgets.FloatSlider(value=15, min=5, max=30, step=0.5, description='Insp Pressure (cmH2O):', style=style)
PEEP_slider = widgets.FloatSlider(value=5, min=0, max=15, step=1, description='PEEP (cmH2O):', style=style)
rate_slider = widgets.IntSlider(value=15, min=8, max=30, step=1, description='Rate (bpm):', style=style)
insp_frac_slider = widgets.FloatSlider(value=0.33, min=0.2, max=0.5, step=0.01, description='I:E ratio:', style=style)
duration_slider = widgets.FloatSlider(value=10, min=5, max=30, step=1, description='Duration (s):', style=style)

# Alarm thresholds
high_pressure_thresh = widgets.FloatSlider(value=25, min=15, max=40, step=1, description='High P alarm (cmH2O):', style=style)
low_Vt_thresh = widgets.FloatSlider(value=300, min=100, max=600, step=10, description='Low Vt alarm (mL):', style=style)
apnea_time = widgets.FloatSlider(value=4, min=2, max=10, step=0.5, description='Apnea time (s):', style=style)

def update_alarms(R, C, P_insp, PEEP, rate, insp_frac, duration,
                  high_P_thresh, low_Vt_thresh, apnea_time_sec):
    clear_output(wait=True)
    t, V, Q, P_aw = simulate_lung(R, C, P_insp, PEEP, rate, insp_frac, duration)
    alarms = {
        'high_pressure_threshold': high_P_thresh,
        'low_Vt_threshold': low_Vt_thresh,
        'apnea_time': apnea_time_sec,
        'high_pressure_trigger': False,
        'low_Vt_trigger': False,
        'apnea_trigger': False,
        'disconnect_trigger': False,
        'high_pressure_msg': '',
        'low_Vt_msg': '',
        'apnea_msg': '',
        'disconnect_msg': ''
    }
    alarms = detect_alarms(t, V, Q, P_aw, rate, alarms)
    plot_with_alarms(t, V, Q, P_aw, R, C, P_insp, PEEP, rate, insp_frac, alarms)

ui = widgets.VBox([
    widgets.HBox([R_slider, C_slider]),
    widgets.HBox([P_insp_slider, PEEP_slider, rate_slider, insp_frac_slider]),
    widgets.HBox([high_pressure_thresh, low_Vt_thresh, apnea_time]),
    duration_slider
])

out = widgets.interactive_output(update_alarms, {
    'R': R_slider, 'C': C_slider, 'P_insp': P_insp_slider, 'PEEP': PEEP_slider,
    'rate': rate_slider, 'insp_frac': insp_frac_slider, 'duration': duration_slider,
    'high_P_thresh': high_pressure_thresh, 'low_Vt_thresh': low_Vt_thresh,
    'apnea_time_sec': apnea_time
})

display(ui, out)

Output()